In [ ]:
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

import re
import gradio as gr
import torch

In [ ]:
repo_id = "eddyfox8812/qwen2-7b-instruct-ai-generated-detector-4000-checkpoint-25"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(repo_id)
model = AutoModelForVision2Seq.from_pretrained(
    repo_id,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
model.eval()

SYSTEM = "당신은 입력 이미지가 'AI 생성 이미지'인지 '사람이 촬영한 실제 사진'인지 판별하는 분류 모델입니다."
PROMPT = """입력으로 이미지가 주어지면, 해당 이미지가 AI가 생성한 이미지인지 여부를 "1" 또는 "0"으로 출력하세요.
- 출력은 반드시 숫자 하나만(문자열/설명 없이) 출력합니다.
  - "0" : AI 생성 이미지(생성형 모델로 만들어졌거나 합성/렌더링/딥페이크 등 인공적으로 생성된 경우 포함)
  - "1" : 실제 사진(사람이 카메라로 촬영한 사진)

판별 시 아래 단서를 종합적으로 고려하세요(단, 어느 한 가지 단서만으로 단정하지 마세요):
1. **질감/디테일의 비현실성**
   - 피부, 머리카락, 천/금속/유리의 질감이 과도하게 매끈하거나 반복 패턴이 보임
   - 미세 디테일(문자, 로고, 손가락, 치아, 눈동자, 귀 등)이 어색하거나 깨짐
2. **조명/그림자/반사의 불일치**
   - 광원 방향과 그림자 방향이 맞지 않음
   - 반사(거울/유리/물)에서 왜곡된 물체 형태 또는 논리적으로 불가능한 반사
3. **기하학/구조의 오류**
   - 배경의 선(창틀, 건물, 도로, 가구)이 비정상적으로 휘거나 이어짐이 부자연스러움
   - 물체 경계가 흐리거나 주변과 섞이는 현상
4. **사진적 특성**
   - 카메라 촬영에서 흔한 노이즈/입자감/심도(아웃포커싱)/모션블러가 자연스럽게 나타나는지
   - 과도하게 선명하거나, 반대로 특정 영역만 비정상적으로 뭉개지는지
5. **전반적 일관성**
   - 장면의 의미/맥락이 자연스러운지, 비현실적 구성 요소가 섞여 있는지

위 기준을 바탕으로 AI 생성으로 판단되면 "0", 실제 사진으로 판단되면 "1"을 출력하십시오.
출력은 오직 숫자 하나(1 또는 0)만 반환하고, 어떠한 추가 문구나 설명도 첨부하지 마십시오.
반드시 출력에 숫자 하나(1 또는 0)가 존재해야합니다
"""

MAX_SIDE = 768

def resize_to_max_side(img: Image.Image, max_side: int = MAX_SIDE) -> Image.Image:
    img = img.convert("RGB")
    w, h = img.size
    s = max_side / max(w, h)
    if s >= 1:
        return img
    return img.resize((int(w*s), int(h*s)), Image.BICUBIC)

# 추론함수
@torch.inference_mode()
def predict(img: Image.Image):
    if img is None:
        return "", "이미지를 업로드하세요."
    img = resize_to_max_side(img)

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [
            {"type": "text", "text": PROMPT},
            {"type": "image", "image": img}
        ]},
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = processor(
        text=[text],
        images=[img],
        padding=True,
        return_tensors="pt",
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    out_ids = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        use_cache=False,
    )

    gen = out_ids[0, inputs["input_ids"].shape[1]:]
    out_text = processor.decode(gen, skip_special_tokens=True).strip()

    m = re.search(r"\b([01])\b", out_text)
    pred_label = m.group(1) if m else out_text

    if pred_label == "0":
        meaning = "해당 사진은 AI로 생성한 사진입니다"
    elif pred_label == "1":
        meaning = "해당 사진은 실제로 촬영된 사진입니다"
    else:
        meaning = f"출력 포맷 오류: {out_text}"

    return pred_label, meaning

# Gradio
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="이미지 업로드"),
    outputs=[
        gr.Textbox(label="예측 라벨(0/1)"),
        gr.Textbox(label="해석"),
    ],
    title="Qwen2-VL AI vs Real 분류 테스트",
    description="0=AI 생성, 1=실제 사진",
)

demo.launch(share=True)

preprocessor_config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://81faa7c9a050894808.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:623: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:601: UserWar